In [ ]:
!nvidia-smi

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from tensorflow.keras.optimizers import Adam, SGD, Optimizer
from tensorflow.keras.losses import SparseCategoricalCrossentropy
import matplotlib.pyplot as plt
import time
import pandas as pd
import numpy as np
import random

In [ ]:
"""
HN_Adam: Hybrid and Adaptive Norming of Adam with AMSGrad — The Modified Algorithm

Implements Algorithm 2, Section 5 from:
    Reyad, M., Sarhan, A. & Arafa, M. "A modified Adam algorithm for deep
    neural network optimization." Neural Comput & Applic 35, 17095-17112 (2023).
    https://doi.org/10.1007/s00521-023-08568-z
"""

class HN_Adam(Optimizer):
    """HN_Adam optimizer (Algorithm 2, Section 5)."""

    def __init__(
        self,
        learning_rate=1e-3,
        beta_1=0.9,
        beta_2=0.999,
        epsilon=1e-8,
        lambda_0=None,
        name="HN_Adam",
        **kwargs,
    ):
        if learning_rate is None:
            raise ValueError("learning_rate cannot be None")
        if learning_rate < 0.0:
            raise ValueError(f"Invalid learning rate: {learning_rate}")
        if beta_1 is None or beta_2 is None:
            raise ValueError("beta_1 and beta_2 cannot be None")
        if not 0.0 <= beta_1 < 1.0:
            raise ValueError(f"Invalid beta_1 value: {beta_1}")
        if not 0.0 <= beta_2 < 1.0:
            raise ValueError(f"Invalid beta_2 value: {beta_2}")
        if epsilon is None or epsilon <= 0.0:
            raise ValueError(f"Invalid epsilon value: {epsilon}")

        # Step 2: Randomly choose Λ_t0 from [2, 4]
        if lambda_0 is None:
            lambda_0 = random.uniform(2.0, 4.0)
        if lambda_0 <= 0.0:
            raise ValueError(f"Invalid lambda_0 value: {lambda_0}, must be > 0")

        super().__init__(learning_rate=learning_rate, name=name, **kwargs)
        self.beta_1 = beta_1
        self.beta_2 = beta_2
        self.epsilon = epsilon
        self.lambda_0 = float(lambda_0)

        self._m = []
        self._v = []
        self._v_hat = []

    def build(self, var_list):
        super().build(var_list)
        if self.built:
            return

        # Step 1: m_0 <- 0, v_0 <- 0, v_hat(0) <- 0
        self._m = [self.add_variable_from_reference(var, "m") for var in var_list]
        self._v = [self.add_variable_from_reference(var, "v") for var in var_list]
        self._v_hat = [self.add_variable_from_reference(var, "v_hat") for var in var_list]

    def update_step(self, gradient, variable, learning_rate=None):
        if gradient is None or variable is None:
            return
        if isinstance(gradient, tf.IndexedSlices):
            raise ValueError("HN_Adam does not support sparse gradients")

        lr = learning_rate if learning_rate is not None else self.learning_rate
        grad = tf.cast(gradient, variable.dtype)
        lr = tf.cast(lr, variable.dtype)
        beta_1 = tf.cast(self.beta_1, variable.dtype)
        beta_2 = tf.cast(self.beta_2, variable.dtype)
        epsilon = tf.cast(self.epsilon, variable.dtype)
        lambda_0 = tf.cast(self.lambda_0, variable.dtype)

        index = self._get_variable_index(variable)
        m = self._m[index]
        v = self._v[index]
        v_hat = self._v_hat[index]

        # Step 5: g_t <- gradient at theta_(t-1)
        abs_grad = tf.abs(grad)

        # Keep m_(t-1) for Steps 7 and 8
        m_prev = tf.identity(m)

        # Step 6: m_t <- beta1 * m_(t-1) + (1 - beta1) * g_t
        m.assign(beta_1 * m + (1.0 - beta_1) * grad)

        # Step 7: m_max <- Max(m_(t-1), |g_t|)
        m_max = tf.maximum(m_prev, abs_grad)
        safe_m_max = tf.where(
            tf.equal(m_max, 0.0),
            tf.ones_like(m_max),
            m_max,
        )

        # Step 8: Lambda(t) <- Lambda_t0 - m_(t-1) / m_max
        lambda_t = lambda_0 - (m_prev / safe_m_max)

        # Step 9: v_t <- beta2 * v_(t-1) + (1 - beta2) * (|g_t| ^ Lambda(t))
        abs_grad_powered = tf.pow(tf.maximum(abs_grad, epsilon), lambda_t)
        v.assign(beta_2 * v + (1.0 - beta_2) * abs_grad_powered)

        # Guard division by zero for 1 / Lambda(t)
        safe_lambda_t = tf.where(
            tf.equal(lambda_t, 0.0),
            tf.fill(tf.shape(lambda_t), epsilon),
            lambda_t,
        )
        inv_lambda_t = 1.0 / safe_lambda_t

        # Step 10: if Lambda(t) < 2 then
        amsgrad_mask = lambda_t < 2.0

        # Step 12: v_hat(t) <- Max(v_hat(t-1), v_t)
        v_hat_candidate = tf.maximum(v_hat, v)
        v_hat.assign(tf.where(amsgrad_mask, v_hat_candidate, v_hat))

        # Step 13: theta_t <- theta_(t-1) - eta * m_t / ((v_hat(t)^(1/Lambda(t))) + epsilon)
        denom_amsgrad = tf.pow(tf.maximum(v_hat, 0.0), inv_lambda_t) + epsilon

        # Step 16: theta_t <- theta_(t-1) - eta * m_t / ((v_t^(1/Lambda(t))) + epsilon)
        denom_adam = tf.pow(tf.maximum(v, 0.0), inv_lambda_t) + epsilon

        # Step 14/17: else branch and end if
        denom = tf.where(amsgrad_mask, denom_amsgrad, denom_adam)

        # Step 18: update theta
        variable.assign_sub(lr * m / denom)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "beta_1": self.beta_1,
                "beta_2": self.beta_2,
                "epsilon": self.epsilon,
                "lambda_0": self.lambda_0,
            }
        )
        return config

In [ ]:
# Build CNN model according to specifications:
# - 2x Conv2D (32 kernels, 3x3) + MaxPool (2x2) + ReLU
# - 2x Conv2D (64 kernels, 3x3) + MaxPool (2x2) + ReLU
# - Flatten to 1D
# - Dense layers: 512, 128, 256, 32 (with ReLU + Dropout 0.1)
# - Output: Dense 10 with Softmax

def create_cnn_model():
    model = models.Sequential([
        # Conv block 1: 2x Conv2D (32 kernels, 3x3)
        layers.Conv2D(32, kernel_size=3, activation='relu', input_shape=(28, 28, 1), padding='valid'),
        layers.Conv2D(32, kernel_size=3, activation='relu', padding='valid'),
        layers.MaxPooling2D(pool_size=(2, 2)),
        
        # Conv block 2: 2x Conv2D (64 kernels, 3x3)
        layers.Conv2D(64, kernel_size=3, activation='relu', padding='valid'),
        layers.Conv2D(64, kernel_size=3, activation='relu', padding='valid'),
        layers.MaxPooling2D(pool_size=(2, 2)),
        
        # Flatten to 1D vector
        layers.Flatten(),
        
        # Fully connected layers: 512, 128, 256, 32 with ReLU and Dropout(0.1)
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.1),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.1),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.1),
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.1),
        
        # Output layer: 10 nodes with Softmax
        layers.Dense(10, activation='softmax')
    ])
    
    return model

# Instantiate model and verify parameter count
model = create_cnn_model()
total_params = model.count_params()
print(f"Total parameters: {total_params}")
assert total_params == 697034, f"Expected 697034 parameters, got {total_params}"
print("✓ Model architecture verified with 697,034 parameters")

In [ ]:
# Load and prepare MNIST dataset
# - 40,000 training samples
# - 10,000 validation samples  
# - 10,000 test samples
# - Float32, normalized to [0, 1]
# - Batch size: 128

print("Loading MNIST dataset...")
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Convert to float32 and normalize to [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Add channel dimension (grayscale)
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

# Split MNIST training set (60,000) into train (40k), val (10k), test (10k)
x_train_split = x_train[:40000]
y_train_split = y_train[:40000]
x_val = x_train[40000:50000]
y_val = y_train[40000:50000]
x_test_split = x_train[50000:60000]
y_test_split = y_train[50000:60000]

print(f"Train samples: {len(x_train_split)}")
print(f"Validation samples: {len(x_val)}")
print(f"Test samples: {len(x_test_split)}")

# Create datasets with batch size 128
batch_size = 128
train_dataset = tf.data.Dataset.from_tensor_slices((x_train_split, y_train_split)).batch(batch_size).shuffle(5000)
val_dataset = tf.data.Dataset.from_tensor_slices((x_val, y_val)).batch(batch_size)
test_dataset = tf.data.Dataset.from_tensor_slices((x_test_split, y_test_split)).batch(batch_size)

# Hyperparameters
num_epochs = 20
learning_rate = 0.001
beta_1 = 0.9
beta_2 = 0.99
epsilon = 1e-8

# Loss function
loss_fn = SparseCategoricalCrossentropy()

# Dictionary to store results for each optimizer
results = {
    'HN_Adam': {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'time': 0, 'min_train_loss': 0, 'test_acc': 0},
    'Adam': {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'time': 0, 'min_train_loss': 0, 'test_acc': 0},
    'AMSGrad': {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'time': 0, 'min_train_loss': 0, 'test_acc': 0},
    'SGD': {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'time': 0, 'min_train_loss': 0, 'test_acc': 0}
}

def evaluate_model(model, dataset):
    """Evaluate model on a dataset and return loss and accuracy."""
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    
    for x_batch, y_batch in dataset:
        predictions = model(x_batch, training=False)
        loss = loss_fn(y_batch, predictions)
        total_loss += loss.numpy()
        
        pred_labels = tf.argmax(predictions, axis=1).numpy()
        total_correct += np.sum(pred_labels == y_batch.numpy())
        total_samples += len(y_batch)
    
    avg_loss = total_loss / len(dataset)
    accuracy = total_correct / total_samples
    return avg_loss, accuracy

def train_with_optimizer(model, optimizer_name, optimizer):
    """Train model with specified optimizer."""
    print(f"\n{'='*60}")
    print(f"Training with {optimizer_name}")
    print(f"{'='*60}")
    
    train_losses, train_accs = [], []
    val_losses, val_accs = [], []
    
    start_time = time.time()
    
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        epoch_correct = 0
        epoch_total = 0
        batch_count = 0
        
        # Training step
        for x_batch, y_batch in train_dataset:
            with tf.GradientTape() as tape:
                predictions = model(x_batch, training=True)
                loss = loss_fn(y_batch, predictions)
            
            gradients = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(gradients, model.trainable_variables))
            
            epoch_loss += loss.numpy()
            pred_labels = tf.argmax(predictions, axis=1).numpy()
            epoch_correct += np.sum(pred_labels == y_batch.numpy())
            epoch_total += len(y_batch)
            batch_count += 1
        
        avg_train_loss = epoch_loss / batch_count
        train_acc = epoch_correct / epoch_total
        train_losses.append(avg_train_loss)
        train_accs.append(train_acc)
        
        # Validation step
        val_loss, val_acc = evaluate_model(model, val_dataset)
        val_losses.append(val_loss)
        val_accs.append(val_acc)
        
        if (epoch + 1) % 5 == 0:
            print(
                f"Epoch {epoch+1}/{num_epochs} | "
                f"Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.4f} | "
                f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}"
            )
    
    elapsed_time = time.time() - start_time
    
    # Test evaluation
    _, test_acc = evaluate_model(model, test_dataset)
    min_train_loss = min(train_losses)
    
    print(f"{optimizer_name} - Test Accuracy: {test_acc:.4f}, Training Time: {elapsed_time:.2f}s")
    
    return train_losses, train_accs, val_losses, val_accs, test_acc, elapsed_time, min_train_loss

# Train with all optimizers
optimizers_config = {
    'HN_Adam': HN_Adam(learning_rate=learning_rate, beta_1=beta_1, beta_2=beta_2, epsilon=epsilon),
    'Adam': Adam(learning_rate=learning_rate, beta_1=beta_1, beta_2=beta_2, epsilon=epsilon),
    'AMSGrad': Adam(learning_rate=learning_rate, beta_1=beta_1, beta_2=beta_2, epsilon=epsilon, amsgrad=True),
    'SGD': SGD(learning_rate=learning_rate)
}

for opt_name, optimizer in optimizers_config.items():
    model = create_cnn_model()
    
    train_loss, train_acc, val_loss, val_acc, test_acc, elapsed_time, min_train_loss = train_with_optimizer(
        model, opt_name, optimizer
    )
    
    results[opt_name]['train_loss'] = train_loss
    results[opt_name]['train_acc'] = train_acc
    results[opt_name]['val_loss'] = val_loss
    results[opt_name]['val_acc'] = val_acc
    results[opt_name]['test_acc'] = test_acc
    results[opt_name]['time'] = elapsed_time
    results[opt_name]['min_train_loss'] = min_train_loss

In [ ]:
# Figure 1: Accuracy curves during training
algorithms = list(results.keys())

plt.figure(figsize=(10, 6))
for alg in algorithms:
    acc_curve = results[alg]['train_acc']
    epochs = range(1, len(acc_curve) + 1)
    plt.plot(epochs, acc_curve, marker='o', linewidth=2, markersize=4, label=alg)

plt.title('Figure 1. Accuracy Curves During Training (CNN)', fontsize=14, fontweight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

# Figure 2: Loss minimization curves during training
plt.figure(figsize=(10, 6))
for alg in algorithms:
    loss_curve = results[alg]['train_loss']
    epochs = range(1, len(loss_curve) + 1)
    plt.plot(epochs, loss_curve, marker='s', linewidth=2, markersize=4, label=alg)

plt.title('Figure 2. Training Loss Minimization Curves (CNN)', fontsize=14, fontweight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Training Loss', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

# Table 1: Summary of results
table_data = []
for alg in algorithms:
    table_data.append({
        'Algorithm': alg,
        'Min Training Loss': f"{results[alg]['min_train_loss']:.6f}",
        'Test Accuracy': f"{results[alg]['test_acc']:.4f}",
        'Training Time (s)': f"{results[alg]['time']:.2f}"
    })

table1 = pd.DataFrame(table_data)
print("\n" + "="*70)
print("Table 1. Minimum Training Loss, Test Accuracy, and Training Time")
print("="*70)
print(table1.to_string(index=False))
print("="*70)